In [41]:
#CELL 0

import numpy as np
from plasmapy.formulary import Debye_length
from astropy import units as u

print("=" * 65)
print("  PLASMA SCREENING SIMULATION — UNIT REFERENCE")
print("=" * 65)

# ── Physical Constants (SI) ───────────────────────────────────
e      = 1.602176634e-19   # [C]        elementary charge
eps0   = 8.8541878128e-12  # [F/m]      vacuum permittivity
m_p    = 1.67262192e-27    # [kg]       proton mass
m_e    = 9.10938372e-31    # [kg]       electron mass
k_B    = 1.380649e-23      # [J/K]      Boltzmann constant
a_0    = 5.29177210e-11    # [m]        Bohr radius
N_A    = 6.02214076e23     # [mol⁻¹]   Avogadro's number

# ── Plasma Conditions ─────────────────────────────────────────
T_eV   = 0.1               # [eV]       electron temperature
T_K    = T_eV * 11604.52   # [K]        electron temperature
n_e    = 1e21              # [m⁻³]      number density

# ── Derived Parameters ────────────────────────────────────────
lambda_D_m  = Debye_length(T_eV * u.eV, n_e * u.m**-3).to(u.m).value  # [m]
lambda_D_cm = lambda_D_m * 1e2          # [cm]
lambda_D_A  = lambda_D_m * 1e10         # [Å]

omega_pe    = np.sqrt(n_e * e**2 / (eps0 * m_e))   # [rad/s]  electron plasma frequency
f_pe        = omega_pe / (2 * np.pi)               # [Hz]     electron plasma frequency
T_pe_s      = 1.0 / f_pe                           # [s]      electron plasma period
T_pe_fs     = T_pe_s * 1e15                        # [fs]

# ── Normalization Scales ───────────────────────────────────────
# The simulation uses these as its base units (= 1 in LAMMPS)
length_scale  = lambda_D_m              # [m]   1 length unit = 1 Debye length
energy_scale  = k_B * T_K              # [J]   1 energy unit = k_BT
mass_scale    = m_p                    # [kg]  1 mass unit   = proton mass
charge_scale  = e                      # [C]   1 charge unit = elementary charge
time_scale    = np.sqrt(m_p * lambda_D_m**2 / (k_B * T_K))  # [s]

# ── Derived Scales ─────────────────────────────────────────────
velocity_scale = length_scale / time_scale          # [m/s]
force_scale    = energy_scale / length_scale        # [N]
pressure_scale = energy_scale / length_scale**3     # [Pa]

# ── Coupling Parameter Γ ──────────────────────────────────────
A_coulomb  = e**2 / (4 * np.pi * eps0)             # [J·m]  Coulomb prefactor
A_norm     = A_coulomb / (k_B * T_K * lambda_D_m)  # [dimensionless]
q_eff      = np.sqrt(A_norm)                        # [dimensionless]
r_WS       = (3 / (4 * np.pi * (150 / (10 * lambda_D_m)**3))) ** (1/3)
Gamma      = A_norm / r_WS                          # [dimensionless]

# WCA core
sigma_ep_norm = a_0 / lambda_D_m                   # [λ_D]  Bohr radius normalized
lj_cut_norm   = 2**(1/6) * sigma_ep_norm           # [λ_D]  WCA cutoff

# Timestep
dt_norm    = T_pe_fs / time_scale / 1e15 / 100     # normalized timestep = T_pe/100

print(f"""
┌─────────────────────────────────────────────────────────────┐
│  NORMALIZATION BASIS                                        │
│  (1 simulation unit = the following physical quantity)      │
├─────────────────────┬───────────────────────────────────────┤
│  Length  [λ_D]      │  {length_scale:.4e} m  =  {lambda_D_A:.4f} Å        │
│  Energy  [k_BT]     │  {energy_scale:.4e} J  =  {energy_scale/e:.6f} eV       │
│  Mass    [m_p]      │  {mass_scale:.4e} kg =  1.007 amu           │
│  Charge  [e]        │  {charge_scale:.4e} C                        │
│  Time    [τ]        │  {time_scale:.4e} s   =  {time_scale*1e12:.4f} ps        │
│  Velocity[λ_D/τ]    │  {velocity_scale:.4e} m/s                   │
│  Force   [k_BT/λ_D] │  {force_scale:.4e} N                        │
│  Pressure[k_BT/λ_D³]│  {pressure_scale:.4e} Pa                    │
├─────────────────────────────────────────────────────────────┤
│  PLASMA PARAMETERS                                          │
├─────────────────────┬───────────────────────────────────────┤
│  Temperature        │  {T_eV} eV  =  {T_K:.2f} K               │
│  Number Density     │  {n_e:.2e} m⁻³                       │
│  Debye Length λ_D   │  {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å        │
│  Electron ω_pe      │  {omega_pe:.4e} rad/s               │
│  Electron f_pe      │  {f_pe:.4e} Hz                      │
│  Electron T_pe      │  {T_pe_s:.4e} s  =  {T_pe_fs:.4f} fs       │
│  Coupling Γ         │  {Gamma:.5f}  (weakly coupled, Γ<<1)     │
├─────────────────────────────────────────────────────────────┤
│  SIMULATION INPUTS (normalized → physical)                  │
├─────────────────────┬───────────────────────────────────────┤
│  Protons            │  150                                  │
│  Electrons          │  150                                  │
│  Box Side (10 λ_D)  │  {10*lambda_D_m:.4e} m  =  {10*lambda_D_A:.2f} Å      │
│  Cutoff  (4  λ_D)   │  {4*lambda_D_m:.4e} m  =  {4*lambda_D_A:.2f} Å       │
│  Timestep (T_pe/100)│  {dt_norm:.6f} τ = {T_pe_fs/100:.4f} fs          │
│  Mass proton        │  1.0 m_p  =  {m_p:.4e} kg            │
│  Mass electron      │  {m_e/m_p:.6f} m_p = {m_e:.4e} kg      │
│  WCA σ (Bohr r.)    │  {sigma_ep_norm:.6f} λ_D = {a_0*1e10:.4f} Å              │
│  WCA ε (= k_BT)     │  1.0 k_BT = {energy_scale/e:.6f} eV         │
│  WCA cutoff         │  {lj_cut_norm:.6f} λ_D = {lj_cut_norm*lambda_D_A:.4f} Å            │
│  Coulomb A (norm.)  │  {A_norm:.6f}  [= e²/4πε₀k_BT λ_D]      │
└─────────────────────┴───────────────────────────────────────┘
""")

  PLASMA SCREENING SIMULATION — UNIT REFERENCE

┌─────────────────────────────────────────────────────────────┐
│  NORMALIZATION BASIS                                        │
│  (1 simulation unit = the following physical quantity)      │
├─────────────────────┬───────────────────────────────────────┤
│  Length  [λ_D]      │  7.4339e-08 m  =  743.3942 Å        │
│  Energy  [k_BT]     │  1.6022e-20 J  =  0.100000 eV       │
│  Mass    [m_p]      │  1.6726e-27 kg =  1.007 amu           │
│  Charge  [e]        │  1.6022e-19 C                        │
│  Time    [τ]        │  2.4019e-11 s   =  24.0194 ps        │
│  Velocity[λ_D/τ]    │  3.0950e+03 m/s                   │
│  Force   [k_BT/λ_D] │  2.1552e-13 N                        │
│  Pressure[k_BT/λ_D³]│  3.8999e+01 Pa                    │
├─────────────────────────────────────────────────────────────┤
│  PLASMA PARAMETERS                                          │
├─────────────────────┬───────────────────────────────────────┤
│  Temp

In [42]:
# CELL 1

# ── Simulation Parameters (normalized units) ──────────────────
# All quantities below are dimensionless multiples of the
# normalization basis printed in Cell 0 above

mass_proton   = 1.0              # [m_p]    proton mass  (= 1.007 amu)
mass_electron = m_e / m_p        # [m_p]    electron mass (= 1/1836 m_p = 5.486e-4 m_p)

charge_proton   = +q_eff         # [e√(k_BT·λ_D/k_e)]  effective normalized proton  charge
charge_electron = -q_eff         # [e√(k_BT·λ_D/k_e)]  effective normalized electron charge

N_protons   = 150                # [count]  number of protons
N_electrons = 150                # [count]  number of electrons
box_size    = 10.0               # [λ_D]    simulation box side length
cutoff      = 4.0                # [λ_D]    Coulomb potential cutoff radius
damping_p   = 0.05               # [τ]      proton   NVT thermostat damping time
damping_e   = 0.005              # [τ]      electron NVT thermostat damping time
steps       = 10000              # [steps]  production run length
dump_freq   = 50                 # [steps]  position output frequency

# Timestep = 1/100th of electron plasma period (standard plasma MD criterion)
dt          = (T_pe_s / time_scale) / 100.0   # [τ]  production timestep

# WCA repulsive core — prevents classical electron-proton collapse
# Physical basis: mimics quantum zero-point pressure / Pauli exclusion
# σ set to Bohr radius (smallest meaningful e-p separation)
sigma_ep  = a_0 / lambda_D_m                  # [λ_D]  LJ σ = Bohr radius
eps_ep    = 1.0                               # [k_BT] LJ ε = thermal energy
lj_cut    = 2**(1/6) * sigma_ep               # [λ_D]  WCA cutoff (purely repulsive)
A_norm    = (e**2 / (4*np.pi*eps0)) / (k_B * T_K * lambda_D_m)  # [dimensionless] Coulomb prefactor

print(f"Normalized Parameters:")
print(f"  Proton mass        : {mass_proton}  [m_p]")
print(f"  Electron mass      : {mass_electron:.8f}  [m_p]  (= m_p/1836)")
print(f"  Charge ±q_eff      : ±{q_eff:.6f}  [normalized charge]")
print(f"  Coulomb prefactor  : {A_norm:.6f}  [dimensionless]")
print(f"  Timestep           : {dt:.8f}  [τ]  =  {dt*time_scale*1e15:.6f} fs")
print(f"  WCA σ              : {sigma_ep:.8f}  [λ_D]  =  {a_0*1e10:.4f} Å")
print(f"  WCA ε              : {eps_ep}  [k_BT]  =  {energy_scale/e:.6f} eV")
print(f"  WCA cutoff         : {lj_cut:.8f}  [λ_D]  =  {lj_cut*lambda_D_A:.4f} Å")
print(f"  Box side           : {box_size}  [λ_D]  =  {box_size*lambda_D_A:.4f} Å")
print(f"  Coulomb cutoff     : {cutoff}  [λ_D]  =  {cutoff*lambda_D_A:.4f} Å")
print(f"  Production dt      : {dt:.8f}  [τ]  =  {T_pe_fs/100:.4f} fs")
print(f"  Total sim time     : {steps*dt*time_scale*1e12:.4f} ps")

Normalized Parameters:
  Proton mass        : 1.0  [m_p]
  Electron mass      : 0.00054462  [m_p]  (= m_p/1836)
  Charge ±q_eff      : ±0.440115  [normalized charge]
  Coulomb prefactor  : 0.193701  [dimensionless]
  Timestep           : 0.00146631  [τ]  =  35.219918 fs
  WCA σ              : 0.00071184  [λ_D]  =  0.5292 Å
  WCA ε              : 1.0  [k_BT]  =  0.100000 eV
  WCA cutoff         : 0.00079901  [λ_D]  =  0.5940 Å
  Box side           : 10.0  [λ_D]  =  7433.9420 Å
  Coulomb cutoff     : 4.0  [λ_D]  =  2973.5768 Å
  Production dt      : 0.00146631  [τ]  =  35.2199 fs
  Total sim time     : 352.1992 ps


In [43]:
#CELL 2

from lammps import lammps

# ── Stable Parameters for 1:100 mass ratio ────────────────────
mass_electron_sim = 1.0 / 100.0    # [m_p]  reduced electron mass (1:100)
sigma_ep          = 0.05           # [λ_D]  WCA σ — min e-p separation (~3.7 Å)
eps_ep            = 5.0            # [k_BT] WCA ε — core strength
lj_cut            = 2**(1/6) * sigma_ep  # [λ_D] WCA cutoff (purely repulsive)

# Timestep: stable for 1:100 mass ratio with WCA
dt_sim = 1e-4                      # [τ]  = {1e-4 * time_scale * 1e15:.4f} fs

lmp = lammps()

lmp.commands_string(f"""
# ══════════════════════════════════════════════════════════════════
#  PLASMA SCREENING SIMULATION — LAMMPS INPUT
#
#  Unit system: normalized (dimensionless LJ-style)
#  See Cell 0 for full SI conversion table.
#
#  Normalization basis:
#    Length  → λ_D  = {lambda_D_m:.4e} m  = {lambda_D_A:.4f} Å
#    Energy  → k_BT = {energy_scale:.4e} J  = {energy_scale/e:.6f} eV
#    Mass    → m_p  = {m_p:.4e} kg
#    Time    → τ    = {time_scale:.4e} s  = {time_scale*1e12:.4f} ps
#    Charge  → q_eff= {q_eff:.6f}
#
#  NOTE: Electron mass reduced to m_p/100 (not physical 1/1836).
#  The full mass ratio requires dt ~ 1e-6 τ — impractical on a
#  laptop. 1:100 is standard in plasma MD literature and correctly
#  reproduces Debye screening statistics. See Cell 0 for details.
# ══════════════════════════════════════════════════════════════════

# ── Unit System ────────────────────────────────────────────────────
units         lj                         # normalized (see Cell 0 for SI conversion)
atom_style    charge                     # id type x[λ_D] y[λ_D] z[λ_D] q[q_eff]
boundary      p p p                      # periodic boundaries x, y, z

# ── Simulation Box ─────────────────────────────────────────────────
# {box_size} λ_D = {box_size*lambda_D_A:.2f} Å = {box_size*lambda_D_m*1e9:.4f} nm per side
region        box block 0 {box_size} 0 {box_size} 0 {box_size}  # [λ_D]
create_box    2 box                      # type 1=proton, type 2=electron

# ── Atom Placement on Regular Grid ─────────────────────────────────
# Grid guarantees zero overlaps at initialization — avoids the
# random placement overlap issues that triggered lost atoms
lattice       sc 1.0                     # simple cubic with spacing 1.0 λ_D
region        proton_region   block 0 {box_size} 0 {box_size/2:.1f} 0 {box_size}  # [λ_D]
region        electron_region block 0 {box_size} {box_size/2:.1f} {box_size} 0 {box_size}  # [λ_D]
create_atoms  1 random {N_protons}   12345 box overlap {lj_cut*2:.5f} maxtry 50000  # protons  [λ_D]
create_atoms  2 random {N_electrons} 67890 box overlap {lj_cut*2:.5f} maxtry 50000  # electrons [λ_D]

# ── Masses [m_p] ───────────────────────────────────────────────────
mass          1 1.000000                 # [m_p] proton   (= {m_p:.4e} kg = 1.007 amu)
mass          2 {mass_electron_sim:.6f}  # [m_p] electron (reduced: m_p/100, physical: m_p/1836)

# ── Charges [q_eff] ────────────────────────────────────────────────
set           type 1 charge +{q_eff:.6f} # [q_eff] proton   charge (= +{e:.3e} C)
set           type 2 charge -{q_eff:.6f} # [q_eff] electron charge (= -{e:.3e} C)

# ── Neighbor List ──────────────────────────────────────────────────
neighbor      0.5 bin                    # [λ_D] neighbor skin distance
neigh_modify  one 5000 delay 0 every 1 check yes  # max 5000 neighbors, check every step
comm_modify   cutoff {cutoff + 1.0:.2f}  # [λ_D] ghost comm range > pair cutoff

# ══════════════════════════════════════════════════════════════════
#  PAIR POTENTIALS (active for all stages after Stage 1)
#
#  hybrid/overlay applies BOTH simultaneously:
#
#  a) coul/cut — bare Coulomb:
#     V(r) = q_i·q_j·A_norm/r  [k_BT]
#     Cutoff = {cutoff} λ_D = {cutoff*lambda_D_A:.2f} Å = {cutoff*lambda_D_m:.3e} m
#
#  b) lj/cut — WCA repulsive core (electron-proton ONLY):
#     V(r) = 4ε[(σ/r)¹²-(σ/r)⁶]+ε  for r < 2^(1/6)σ, else 0
#     σ = {sigma_ep} λ_D = {sigma_ep*lambda_D_A:.3f} Å  (min e-p separation)
#     ε = {eps_ep} k_BT = {eps_ep*energy_scale/e:.4f} eV
#     Prevents classical e-p collapse — mimics quantum
#     zero-point pressure and Pauli exclusion principle
# ══════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
#  STAGE 1: Soft push-off (no charges, safe initialization)
#  V_soft(r) = A[1 + cos(π·r/r_cut)]  [k_BT]
# ══════════════════════════════════════════════════════════════════
pair_style    soft {lj_cut*4:.5f}        # [λ_D] soft cutoff = 4× WCA cutoff
pair_coeff    * * 100.0                  # [k_BT] soft amplitude

# Disable charges during soft push-off to avoid Coulomb explosions
set           type 1 charge 0.0          # temporarily zero proton  charge [q_eff]
set           type 2 charge 0.0          # temporarily zero electron charge [q_eff]

fix           s1 all nve/limit 0.001     # [λ_D] hard displacement cap per step
timestep      0.00001                    # [τ]   = {0.00001*time_scale*1e15:.4f} fs
run           3000                       # soft push-off steps [steps]
unfix         s1

# Restore real charges
set           type 1 charge +{q_eff:.6f} # [q_eff] restore proton  charge
set           type 2 charge -{q_eff:.6f} # [q_eff] restore electron charge

# ══════════════════════════════════════════════════════════════════
#  STAGE 2: Switch on Coulomb + WCA, minimize under real forces
#  Minimization places atoms at proper separations BEFORE
#  any velocities are assigned — eliminates first-step explosions
# ══════════════════════════════════════════════════════════════════
pair_style    hybrid/overlay coul/cut {cutoff:.4f} lj/cut {lj_cut:.6f}  # [λ_D] [λ_D]

pair_coeff    1 1 coul/cut               # proton  -proton  : repulsive Coulomb
pair_coeff    1 2 coul/cut               # proton  -electron: attractive Coulomb
pair_coeff    2 2 coul/cut               # electron-electron: repulsive Coulomb

pair_coeff    1 2 lj/cut {eps_ep:.4f} {sigma_ep:.6f} {lj_cut:.6f}  # e-p WCA [k_BT][λ_D][λ_D]
pair_coeff    1 1 lj/cut 0.0 {sigma_ep:.6f} 0.0  # p-p: WCA off [k_BT][λ_D][λ_D]
pair_coeff    2 2 lj/cut 0.0 {sigma_ep:.6f} 0.0  # e-e: WCA off [k_BT][λ_D][λ_D]

minimize      1.0e-6 1.0e-8 10000 100000  # etol[k_BT] ftol[k_BT/λ_D] maxiter maxeval

# ══════════════════════════════════════════════════════════════════
#  STAGE 3: langevin + nve/limit warm-up (valid combination)
#  fix langevin = thermostat only (NOT an integrator)
#  fix nve/limit = integrator WITH hard displacement cap
#  Temperature ramp: 1% → 10% → 50% → 100% of target T
# ══════════════════════════════════════════════════════════════════
group         protons   type 1           # all protons   (type 1)
group         electrons type 2           # all electrons (type 2)

reset_timestep 0
velocity      all create 0.01 11111 dist gaussian  # [k_BT/k_B] 1% of target T

# 1% T
fix           l1 all langevin 0.01 0.01 0.1 11111  # [k_BT/k_B][k_BT/k_B][τ] thermostat
fix           n1 all nve/limit 0.0005    # [λ_D] hard displacement cap
timestep      {dt_sim*0.01:.8f}          # [τ]   1% of production timestep
run           2000                       # [steps]
unfix         l1
unfix         n1

# 10% T
fix           l2 all langevin 0.01 0.1 0.1 22222   # [k_BT/k_B][k_BT/k_B][τ]
fix           n2 all nve/limit 0.001     # [λ_D]
timestep      {dt_sim*0.05:.8f}          # [τ]   5% of production timestep
run           2000                       # [steps]
unfix         l2
unfix         n2

# 50% T
fix           l3 all langevin 0.1 0.5 0.1 33333    # [k_BT/k_B][k_BT/k_B][τ]
fix           n3 all nve/limit 0.005     # [λ_D]
timestep      {dt_sim*0.2:.8f}           # [τ]   20% of production timestep
run           2000                       # [steps]
unfix         l3
unfix         n3

# 100% T
fix           l4 all langevin 0.5 1.0 0.1 44444    # [k_BT/k_B][k_BT/k_B][τ]
fix           n4 all nve/limit 0.01      # [λ_D]
timestep      {dt_sim*0.5:.8f}           # [τ]   50% of production timestep
run           3000                       # [steps]
unfix         l4
unfix         n4

# ══════════════════════════════════════════════════════════════════
#  STAGE 4: Production Run (full NVT, separate thermostats)
#  Separate per-species thermostats required: at same T,
#  electrons (m_p/100) move ~10× faster than protons
# ══════════════════════════════════════════════════════════════════
reset_timestep 0

velocity      protons   create 1.0 55555 dist gaussian  # [k_BT/k_B] proton  Maxwell-Boltzmann
velocity      electrons create 1.0 66666 dist gaussian  # [k_BT/k_B] electron Maxwell-Boltzmann

fix           1 protons   nvt temp 1.0 1.0 0.5    # [k_BT/k_B][k_BT/k_B][τ] proton  NVT
fix           2 electrons nvt temp 1.0 1.0 0.05   # [k_BT/k_B][k_BT/k_B][τ] electron NVT
fix           3 all momentum 100 linear 1 1 1      # remove COM drift every 100 steps

# ── Dump Positions ─────────────────────────────────────────────────
dump          1 all custom {dump_freq} plasma.dump id type x y z   # positions [λ_D]
dump_modify   1 sort id                                             # sort by atom id

# ── Thermo Output ──────────────────────────────────────────────────
# T[k_BT/k_B] PE[k_BT] KE[k_BT] E_total[k_BT] P[k_BT/λ_D³]
thermo_style  custom step temp pe ke etotal press
thermo        {dump_freq}                # print every {dump_freq} steps

# ── Production Run ─────────────────────────────────────────────────
timestep      {dt_sim:.8f}              # [τ] = {dt_sim*time_scale*1e15:.4f} fs
run           {steps}                   # {steps} steps = {steps*dt_sim*time_scale*1e12:.4f} ps
""")

lmp.close()
print(f"✓ LAMMPS simulation complete")
print(f"  Mass ratio used      : 1:100 (physical: 1:1836)")
print(f"  Timestep             : {dt_sim:.6f} τ  =  {dt_sim*time_scale*1e15:.4f} fs")
print(f"  Total simulated time : {steps*dt_sim*time_scale*1e12:.4f} ps")
print(f"  Frames written       : {steps // dump_freq}")

LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Created orthogonal box = (0 0 0) to (10 10 10)
  1 by 1 by 1 MPI processor grid
Lattice spacing in x,y,z = 1 1 1
Created 150 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.000 seconds
Created 150 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.000 seconds
Setting atom values ...
  150 settings made for charge
Setting atom values ...
  150 settings made for charge
Setting atom values ...
  150 settings made for charge
Setting atom values ...
  150 settings made for charge
Generated 0 of 1 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 5000, page size: 100000
  master list distance cutoff = 0.72449
  ghost atom cutoff = 5
  binsize = 0.362245, bins = 28 28

In [44]:
# CELL 3

def parse_lammps_dump(filepath):
    frames_pos   = []
    frames_types = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            i += 2
            i += 2
            i += 4  # skip box bounds
            i += 1  # skip ITEM: ATOMS header

            positions = []
            types     = []
            while i < len(lines) and "ITEM:" not in lines[i]:
                parts = lines[i].split()
                types.append(int(parts[1]))
                x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                positions.append([x, y, z])
                i += 1

            frames_pos.append(np.array(positions))
            frames_types.append(np.array(types))
        else:
            i += 1

    return frames_pos, frames_types

frames_pos, frames_types = parse_lammps_dump("plasma.dump")

print(f"✓ Parsed {len(frames_pos)} frames")
print(f"✓ Protons   per frame : {np.sum(frames_types[0] == 1)}")
print(f"✓ Electrons per frame : {np.sum(frames_types[0] == 2)}")

✓ Parsed 201 frames
✓ Protons   per frame : 150
✓ Electrons per frame : 150


In [45]:
#CELL 4

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
import numpy as np

# ── Variable aliases (renamed in Cell 0 / Cell 2) ─────────────
lambda_D = lambda_D_m          # [m]   Debye length
gamma    = Gamma               # [dimensionless] coupling parameter
timestep = dt_sim              # [τ]   production timestep
mass_electron = mass_electron_sim  # [m_p] reduced electron mass

# ── Skip frame 0 (pre-run initial state) ──────────────────
frames_pos_use   = frames_pos[1:]
frames_types_use = frames_types[1:]
n_frames         = len(frames_pos_use)

# ── RDF Setup ─────────────────────────────────────────────
n_bins    = 80
r_max     = cutoff * 0.9
r_bins    = np.linspace(0.1, r_max, n_bins + 1)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
rdf_accum = np.zeros(n_bins)
rdf_count = 0

# ── Radial Ring Image Setup ────────────────────────────────
ring_res  = 300
ring_half = ring_res // 2
yy, xx    = np.mgrid[-ring_half:ring_half, -ring_half:ring_half]
pixel_r   = np.sqrt(xx**2 + yy**2) / ring_half * r_max

# ── Minimum Image Convention ───────────────────────────────
def min_image(delta, box):
    return delta - box * np.round(delta / box)

# ── Precompute RDF across all frames ──────────────────────
rdf_history = []

print("Precomputing RDF across all frames...")
for frame_idx in range(n_frames):
    pos   = frames_pos_use[frame_idx]
    types = frames_types_use[frame_idx]

    p_pos = pos[types == 1]
    e_pos = pos[types == 2]

    frame_rdf = np.zeros(n_bins)
    for p in p_pos:
        for e in e_pos:
            dr   = min_image(e - p, box_size)
            dist = np.linalg.norm(dr)
            if 0.1 < dist < r_max:
                bin_idx = np.searchsorted(r_bins, dist) - 1
                if 0 <= bin_idx < n_bins:
                    frame_rdf[bin_idx] += 1

    n_e_density = N_electrons / box_size**3
    for b in range(n_bins):
        shell_vol = (4/3) * np.pi * (r_bins[b+1]**3 - r_bins[b]**3)
        expected  = n_e_density * shell_vol * N_protons
        frame_rdf[b] = frame_rdf[b] / expected if expected > 0 else 0

    rdf_accum += frame_rdf
    rdf_count += 1
    rdf_history.append(rdf_accum.copy() / rdf_count)

print(f"✓ Precomputed {n_frames} frames")

# ── Build radial ring images ───────────────────────────────
def make_ring_image(rdf):
    img = np.zeros((ring_res, ring_res))
    for i in range(ring_res):
        for j in range(ring_res):
            r = pixel_r[i, j]
            if r < r_max:
                bin_idx = np.searchsorted(r_bins, r) - 1
                if 0 <= bin_idx < n_bins:
                    img[i, j] = rdf[bin_idx]
    return img

print("Building radial ring images...")
ring_history = [make_ring_image(rdf_history[i]) for i in range(n_frames)]
print("✓ Ring images ready")

# ── Derived stats for the panel ───────────────────────────
r_avg      = (3 / (4 * np.pi * (N_protons / box_size**3))) ** (1/3)
gamma      = A_norm / r_avg
plasma_freq = np.sqrt(N_protons / box_size**3 / mass_proton)  # normalized
rdf_final  = rdf_history[-1]
peak_val   = rdf_final.max()
peak_r     = r_centers[np.argmax(rdf_final)]

# ── Build Figure Layout ───────────────────────────────────
fig = plt.figure(figsize=(15, 10), facecolor="#0a0a1a")

# Outer grid: top (simulation panels) + bottom (stats bar)
outer_gs = gridspec.GridSpec(2, 1,
                             height_ratios=[5, 1],
                             hspace=0.08,
                             figure=fig)

# Top section: 3D view + RDF + ring
top_gs = gridspec.GridSpecFromSubplotSpec(2, 2,
                                          subplot_spec=outer_gs[0],
                                          width_ratios=[1.3, 1],
                                          hspace=0.45, wspace=0.35)

ax3d    = fig.add_subplot(top_gs[:, 0], projection='3d')
ax_rdf  = fig.add_subplot(top_gs[0, 1])
ax_ring = fig.add_subplot(top_gs[1, 1])

# Bottom section: stats panel
ax_stats = fig.add_subplot(outer_gs[1])
ax_stats.set_facecolor("#0d0d1f")
ax_stats.axis('off')
for spine in ax_stats.spines.values():
    spine.set_edgecolor('#334')

for ax in [ax_rdf, ax_ring]:
    ax.set_facecolor("#0d0d2b")
    for spine in ax.spines.values():
        spine.set_edgecolor('#444466')
    ax.tick_params(colors='#aaaacc', labelsize=7)
    ax.xaxis.label.set_color('#aaaacc')
    ax.yaxis.label.set_color('#aaaacc')
    ax.title.set_color('white')

# ── Initialize ring image ─────────────────────────────────
ring_img = ax_ring.imshow(
    gaussian_filter(ring_history[0], sigma=3),
    extent=[-r_max, r_max, -r_max, r_max],
    origin='lower', cmap='inferno',
    aspect='equal', vmin=0, vmax=2.5
)
theta = np.linspace(0, 2 * np.pi, 300)
ax_ring.plot(np.cos(theta), np.sin(theta),
             color='cyan', linewidth=0.8,
             linestyle='--', alpha=0.6, label='r = 1 λ_D')
ax_ring.legend(fontsize=6, facecolor='#1a1a2e',
               labelcolor='white', framealpha=0.7, loc='upper right')

cbar = fig.colorbar(ring_img, ax=ax_ring, fraction=0.046, pad=0.04)
cbar.ax.tick_params(colors='#aaaacc', labelsize=6)
cbar.set_label('g(r)', color='#aaaacc', fontsize=7)

# ── Stats panel content ───────────────────────────────────
col_style = dict(color='#aaaacc', fontsize=8,
                 fontfamily='monospace', transform=ax_stats.transAxes,
                 va='center')
header_style = dict(color='white', fontsize=8, fontweight='bold',
                    fontfamily='monospace', transform=ax_stats.transAxes,
                    va='center')

# Divider line at top of stats panel
ax_stats.axhline(y=0.95, color='#334477', linewidth=0.8)

# Column headers
ax_stats.text(0.01,  0.72, "── PARTICLES ──",         **header_style)
ax_stats.text(0.21,  0.72, "── PLASMA CONDITIONS ──",  **header_style)
ax_stats.text(0.50,  0.72, "── DEBYE PARAMETERS ──",   **header_style)
ax_stats.text(0.74,  0.72, "── SIMULATION ──",         **header_style)

# Column 1 — Particles
ax_stats.text(0.01, 0.45, f"Protons        : {N_protons}",     **col_style)
ax_stats.text(0.01, 0.20, f"Electrons      : {N_electrons}",   **col_style)

# Column 2 — Plasma conditions
ax_stats.text(0.21, 0.45, f"Temperature    : {T_eV} eV  ({T_K:.0f} K)",  **col_style)
ax_stats.text(0.21, 0.20, f"Density        : {n_e:.2e} m⁻³",             **col_style)

# Column 3 — Debye parameters
ax_stats.text(0.50, 0.45, f"Debye Length   : {lambda_D:.4e} m",           **col_style)
ax_stats.text(0.50, 0.20, f"Coupling Γ     : {gamma:.5f}  (weakly coupled)" if gamma < 1
                           else f"Coupling Γ     : {gamma:.5f}  (strongly coupled)", **col_style)

# Column 4 — Simulation inputs
ax_stats.text(0.74, 0.45, f"MD Steps       : {steps}  |  Timestep : {timestep}",  **col_style)
ax_stats.text(0.74, 0.20, f"Box Size       : {box_size} λ_D  |  Cutoff : {cutoff} λ_D", **col_style)

# ── Dynamic stats (update each frame) ────────────────────
rdf_peak_text = ax_stats.text(0.50, -0.15, "", fontsize=8, color='mediumpurple',
                               fontfamily='monospace', transform=ax_stats.transAxes,
                               va='center', ha='center', clip_on=False)

def update(frame_idx):
    # ── Left: 3D particle positions ───────────────────────
    ax3d.cla()
    ax3d.set_facecolor("#0a0a1a")

    pos   = frames_pos_use[frame_idx]
    types = frames_types_use[frame_idx]
    p_pos = pos[types == 1]
    e_pos = pos[types == 2]

    ax3d.scatter(p_pos[:, 0], p_pos[:, 1], p_pos[:, 2],
                 c='crimson', s=60, alpha=0.95,
                 edgecolors='darkred', linewidths=0.4, label='Protons')
    ax3d.scatter(e_pos[:, 0], e_pos[:, 1], e_pos[:, 2],
                 c='dodgerblue', s=10, alpha=0.7,
                 edgecolors='navy', linewidths=0.2, label='Electrons')

    ax3d.set_xlim(0, box_size); ax3d.set_ylim(0, box_size); ax3d.set_zlim(0, box_size)
    ax3d.set_xlabel("x (λ_D)", color='white', fontsize=7)
    ax3d.set_ylabel("y (λ_D)", color='white', fontsize=7)
    ax3d.set_zlabel("z (λ_D)", color='white', fontsize=7)
    ax3d.tick_params(colors='white', labelsize=6)
    ax3d.set_facecolor("#0a0a1a")
    ax3d.legend(loc='upper left', fontsize=7,
                facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)
    ax3d.set_title(
        f"Particle Positions  |  {N_protons}p + {N_electrons}e\n"
        f"Frame {frame_idx + 1}/{n_frames}",
        color='white', fontsize=9)

    # ── Right top: Running RDF ─────────────────────────────
    ax_rdf.cla()
    ax_rdf.set_facecolor("#0d0d2b")
    rdf = rdf_history[frame_idx]
    ax_rdf.plot(r_centers, rdf, color='mediumpurple', linewidth=1.5)
    ax_rdf.axhline(y=1.0, color='gray', linestyle='--',
                   linewidth=0.8, alpha=0.6, label='Random (g=1)')
    ax_rdf.fill_between(r_centers, 1.0, rdf,
                        where=(rdf > 1), alpha=0.25,
                        color='mediumpurple', label='Electron excess')
    ax_rdf.fill_between(r_centers, 1.0, rdf,
                        where=(rdf < 1), alpha=0.25,
                        color='tomato', label='Electron deficit')
    ax_rdf.set_xlim(0, r_max)
    ax_rdf.set_ylim(0, max(2.5, rdf.max() * 1.1))
    ax_rdf.set_xlabel("r (λ_D)", color='#aaaacc', fontsize=8)
    ax_rdf.set_ylabel("g(r)  electron-proton", color='#aaaacc', fontsize=8)
    ax_rdf.set_title("Radial Distribution Function\n(accumulating over time)",
                     color='white', fontsize=8)
    ax_rdf.legend(fontsize=6, facecolor='#1a1a2e',
                  labelcolor='white', framealpha=0.7)
    ax_rdf.tick_params(colors='#aaaacc', labelsize=7)
    for spine in ax_rdf.spines.values():
        spine.set_edgecolor('#444466')

    # ── Right bottom: Radial ring image ───────────────────
    smoothed = gaussian_filter(ring_history[frame_idx], sigma=3)
    ring_img.set_data(smoothed)
    ring_img.set_clim(vmin=0, vmax=max(smoothed.max(), 2.5))
    ax_ring.set_xlabel("Δx (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_ylabel("Δy (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_title(
        "Debye Screening Cloud\n(radial electron density, time-averaged)",
        color='white', fontsize=8)

    # ── Stats panel: update live RDF peak ─────────────────
    current_peak = rdf_history[frame_idx].max()
    current_peak_r = r_centers[np.argmax(rdf_history[frame_idx])]
    rdf_peak_text.set_text(
        f"[ Live ]  RDF Peak:  g(r) = {current_peak:.3f}  |  "
        f"Peak at r = {current_peak_r:.3f} λ_D  |  "
        f"Frame {frame_idx + 1}/{n_frames}"
    )

    fig.patch.set_facecolor("#0a0a1a")

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=60)

writer = animation.FFMpegWriter(fps=20, bitrate=2400)
ani.save("plasma_screening.mp4", writer=writer)
plt.close()
print("✓ Saved plasma_screening.mp4")

Precomputing RDF across all frames...
✓ Precomputed 200 frames
Building radial ring images...
✓ Ring images ready
✓ Saved plasma_screening.mp4
